# GaMMA association of PhaseNet-DAS picks (run that produced the 946-event catalogue)

Reconstructed 2026-11-02 from the IPython kernel history (`~/.ipython/profile_default/history.sqlite`,
session 1701, first pass, executed 2026-10-29 16:25; `gamma_picks.csv` written 2026-10-29 17:11).
The same kernel was then re-run from the top with `min_p_picks_per_eq = min_s_picks_per_eq = 50`
(the `*_bothwaves` run, 431 events, finished 2026-10-30 10:15), and that second pass edited
`gamma_phasenet_das_rainier.ipynb` in place and overwrote `gamma_events.csv`. This file keeps the
first pass. The cells below are the exact cells that were executed, in order.

Outputs (in `.../gamma_association_10282026/rainier_mora_das/`):
`gamma_picks.csv` (758,167 picks, 217,292 assigned to 946 events; unchanged on disk) and
`gamma_events.csv` (overwritten by the second pass; the 946-event table was rebuilt from
`gamma_picks.csv` as `gamma_events_restored946.csv`, 2026-10-30 10:53).

Settings that differ from `gamma_phasenet_das_rainier.ipynb`:

| item | this run (946 events) | `gamma_phasenet_das_rainier.ipynb` (431 events) |
| --- | --- | --- |
| `min_p_picks_per_eq` / `min_s_picks_per_eq` | 0 / 0 (no P/S requirement, as in the PyOcto `events_octo_das` run) | 50 / 50 |
| picks output | `gamma_picks.csv` | `gamma_picks_bothwaves.csv` |

Everything else is identical: every 10th located channel (320 stations), `min_picks_per_eq = 100`,
BGMM with oversample 5, `dbscan_eps = 10 s`, IASP91 eikonal at h = 1 km, box lon -123 to -120,
lat 45 to 48, depth 0-40 km, GaMMA 1.2.17 in the `test_environment` kernel. Took 1928 s with 16 CPUs
for the second pass; the first pass was of the same order.

**Re-running the association cell overwrites `gamma_events.csv` and `gamma_picks.csv` in the results folder.**


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pyproj import Proj
from gamma.utils import association, estimate_eps

## 1. Data

In [2]:
## Paths: where the inputs live and where results and figures go
region = "rainier_mora_das"

run_dir = ("/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/"
           "phasenet-3-month-run_08182026")
picks_dir = os.path.join(run_dir, "results", "picks_phasenet_das")   # decimator_YYYY-MM-DD_HH.MM.SS_UTC.csv
data_path = lambda x: os.path.join("data", x)
station_csv = data_path("Mora_channellocation.csv")
iasp91_csv = data_path("vmodel/IASP91_1.csv")
pnsn_csv = data_path("pnsn_catalog_aug_nov.csv")
comcat_csv = data_path("comcat_rainier_25km.csv")

result_root = ("/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/"
               f"gamma_association_11022026/{region}")
os.makedirs(result_root, exist_ok=True)
result_path = lambda x: os.path.join(result_root, x)
os.makedirs("figures", exist_ok=True)
figure_dir = lambda x: os.path.join("figures", x)

for name, p in [("picks", picks_dir), ("stations", station_csv), ("iasp91", iasp91_csv),
                ("pnsn", pnsn_csv), ("comcat", comcat_csv), ("results", result_root)]:
    print(f"{'ok' if os.path.exists(p) else 'MISSING':8s}{name:10s}{p}")

## Inventory of the pick files. The file name carries the start time of each 1-minute block, so a
## time window can be selected without opening all 126k files. Per-file pick counts come from the
## run's own file summary, which has already read every file.
summary = pd.read_csv(os.path.join(run_dir, "DAS_catalog_results", "DAS_file_summary.csv"),
                      usecols=["source_file", "input_data_rows"])
pick_files = pd.DataFrame({
    "file": summary["source_file"],
    "t_start": pd.to_datetime(summary["source_file"].str[len("decimator_"):-len("_UTC.csv")],
                              format="%Y-%m-%d_%H.%M.%S", utc=True),
    "n_picks": summary["input_data_rows"].fillna(0).astype(int),
}).sort_values("t_start").set_index("t_start")

print(f"\n{len(pick_files)} pick files, {pick_files.index.min()} to {pick_files.index.max()}")
print(f"{(pick_files['n_picks'] > 0).sum()} files with picks, {pick_files['n_picks'].sum():,} picks in total "
      f"(max {pick_files['n_picks'].max():,} in one minute)")
gap = pick_files.index.to_series().diff()
gaps = gap[gap > pd.Timedelta(minutes=2)]
print(f"{len(gaps)} gaps longer than 2 min" + (f"; largest {gaps.max()} ending {gaps.idxmax()}" if len(gaps) else ""))

ok      picks     /1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/phasenet-3-month-run_08182026/results/picks_phasenet_das
ok      stations  data/Mora_channellocation.csv
ok      iasp91    data/vmodel/IASP91_1.csv
ok      pnsn      data/pnsn_catalog_aug_nov.csv
ok      comcat    data/comcat_rainier_25km.csv
ok      results   /1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/gamma_association_11022026/rainier_mora_das

126452 pick files, 2023-08-23 19:19:28+00:00 to 2023-11-21 23:59:00+00:00
57954 files with picks, 13,795,940 picks in total (max 13,158 in one minute)
3 gaps longer than 2 min; largest 1 days 05:08:26 ending 2023-09-19 12:56:26+00:00


In [3]:
## Time window. The first 24 hours were used as a test (outputs kept in result_root/test_first_24h);
## now the full run.
import time
t_start = pick_files.index.min()
t_end = pick_files.index.max() + pd.Timedelta(minutes=1)      # full run
# t_end = t_start + pd.Timedelta(hours=24)                      # 24-hour test
window_files = pick_files[(pick_files.index >= t_start) & (pick_files.index < t_end)]
print(f"window {t_start} .. {t_end}: {len(window_files)} files, {window_files['n_picks'].sum():,} picks")

## read stations: every DAS channel with a surveyed position is a station. Neighbouring channels are
## 9.6 m apart and see the same wavefield, so we keep every `channel_stride`-th located channel.
channel_stride = 10
geom = pd.read_csv(station_csv)
stations = pd.DataFrame({
    "station_id": geom["Channel"].astype(int),
    "longitude": geom["Longitude [°]"],
    "latitude": geom["Latitude [°]"],
    "elevation_m": geom["Elevation [m]"],
}).dropna(subset=["longitude", "latitude"])
stations = stations[stations["station_id"] % channel_stride == 0].reset_index(drop=True)
stations.to_csv(result_path("stations.csv"), index=False)
stations.rename(columns={"station_id": "id"}, inplace=True)
stations["id"] = stations["id"].astype(str)
print(f"{len(stations)} DAS stations (stride {channel_stride}) out of {geom['Latitude [°]'].notna().sum()} located channels")
print("Station format:", stations.iloc[:5], sep="\n")

## read picks: one PhaseNet-DAS CSV per minute, columns channel_index, phase_index, phase_time,
## phase_score, phase_type. Only picks on channels in the station table are kept.
## Files with no picks (header only) are skipped; each file is reduced to the station channels before
## concatenating, so the full 13.8 M-pick run does not have to sit in memory at once.
t0 = time.time()
station_channels = set(stations["id"].astype(int))
chunks = []
for f in window_files.loc[window_files["n_picks"] > 0, "file"]:
    df = pd.read_csv(os.path.join(picks_dir, f))
    df = df[df["channel_index"].isin(station_channels)]
    if len(df):
        chunks.append(df)
picks = pd.concat(chunks, ignore_index=True)
del chunks
print(f"read {int((window_files['n_picks'] > 0).sum())} pick files in {time.time() - t0:.0f} s")
picks["phase_time"] = pd.to_datetime(picks["phase_time"], utc=True, format="ISO8601")
picks.rename(columns={"channel_index": "station_id"}, inplace=True)
picks.to_csv(result_path("picks.csv"), index=False, date_format="%Y-%m-%dT%H:%M:%S.%f")
picks.rename(columns={"station_id": "id", "phase_time": "timestamp", "phase_type": "type", "phase_score": "prob"}, inplace=True)
picks["id"] = picks["id"].astype(str)
print(f"\n{len(picks)} picks on the station channels: " + ", ".join(f"{k} {v}" for k, v in picks["type"].value_counts().items()))
print("Pick format:", picks.iloc[:5], sep="\n")

## Region. The example derives the box from the station extent, but a single fiber is a 20 km line
## and the Rainier seismicity sits 10-15 km north of it, so the box is given explicitly: the same
## lat 45-48, lon -123 to -120, 0-40 km box as the PyOcto `events_octo_das` run (~230 x 330 km).
x0 = stations["longitude"].median()
y0 = stations["latitude"].median()
config = {}
config["center"] = (x0, y0)
config["xlim_degree"] = (-123, -120)
config["ylim_degree"] = (45, 48)


## projection to km
proj = Proj(f"+proj=aeqd +lon_0={config['center'][0]} +lat_0={config['center'][1]} +units=km")
stations[["x(km)", "y(km)"]] = stations.apply(lambda x: pd.Series(proj(longitude=x.longitude, latitude=x.latitude)), axis=1)
stations["z(km)"] = stations["elevation_m"].apply(lambda x: -x / 1e3)

### setting GMMA configs
config["use_dbscan"] = True
config["use_amplitude"] = False           # PhaseNet-DAS picks carry no amplitude
config["method"] = "BGMM"
if config["method"] == "BGMM":  ## BayesianGaussianMixture
    config["oversample_factor"] = 5
if config["method"] == "GMM":  ## GaussianMixture
    config["oversample_factor"] = 1
#!!Prior for (time and amplitude): set a larger value to prevent splitting events. A too large value will cause merging events.
# config["covariance_prior"] = [5.0, 2.0]

# earthquake location
config["vel"] = {"p": 6.0, "s": 6.0 / 1.75}       # only used for DBSCAN scaling once eikonal is set
config["dims"] = ["x(km)", "y(km)", "z(km)"]
config["x(km)"] = proj(longitude=config["xlim_degree"], latitude=[config["center"][1]] * 2)[0]
config["y(km)"] = proj(longitude=[config["center"][0]] * 2, latitude=config["ylim_degree"])[1]
config["z(km)"] = (0, 40)
config["bfgs_bounds"] = (
    (config["x(km)"][0] - 1, config["x(km)"][1] + 1),  # x
    (config["y(km)"][0] - 1, config["y(km)"][1] + 1),  # y
    (0, config["z(km)"][1] + 1),  # z
    (None, None),  # t
)

# DBSCAN:
##!!Truncate the picks into segments: change the dbscan_eps to balance speed and event splitting.
## estimate_eps() scales with the nearest-station spacing, which for DAS channels 96 m apart gives a
## few hundredths of a second and would split every event, so eps is set by hand: 10 s spans the
## whole P-to-S moveout along the 20 km fiber for a local event.
print(f"\nestimate_eps would give {estimate_eps(stations, config['vel']['p']):.3f} s; using 10 s")
config["dbscan_eps"] = 10.0
config["dbscan_min_samples"] = 3

## using Eikonal for 1D velocity model: IASP91 down to the bottom of the search depth
velocity_model = pd.read_csv(iasp91_csv)                      # columns depth, r, vp, vs
velocity_model = velocity_model[velocity_model["depth"] <= config["z(km)"][1]].copy()
# IASP91 repeats the depth at layer boundaries; np.interp needs increasing depths, so nudge duplicates
velocity_model["depth"] += velocity_model.groupby("depth").cumcount() * 1e-3
vel = {"z": velocity_model["depth"].values, "p": velocity_model["vp"].values, "s": velocity_model["vs"].values}
h = 1.0
config["eikonal"] = {"vel": vel, "h": h, "xlim": config["x(km)"], "ylim": config["y(km)"], "zlim": config["z(km)"]}

# set number of cpus
config["ncpu"] = min(16, os.cpu_count())

##!!Post filtering (indepent of gmm): change these parameters to filter out associted picks with large errors
config["min_picks_per_eq"] = 100          # ~ EQNet's 500 at stride 1, scaled to stride 10
config["min_p_picks_per_eq"] = 0
config["min_s_picks_per_eq"] = 0
config["max_sigma11"] = 2.0  # second
config["max_sigma22"] = 1.0  # log10(m/s)
config["max_sigma12"] = 1.0  # covariance

for k, v in config.items():
    if k == "eikonal":
        print(f"{k}: IASP91 {len(vel['z'])} layers, h={h} km, zlim={v['zlim']}")
    else:
        print(f"{k}: {v}")

window 2023-08-23 19:19:28+00:00 .. 2023-11-22 00:00:00+00:00: 126452 files, 13,795,940 picks
320 DAS stations (stride 10) out of 3191 located channels
Station format:
   id  longitude  latitude  elevation_m
0   0 -121.73694  46.78586       1667.0
1  10 -121.73703  46.78563       1668.0
2  20 -121.73825  46.78545       1665.0
3  30 -121.73922  46.78498       1662.0
4  40 -121.73973  46.78431       1658.0
read 57954 pick files in 633 s

758167 picks on the station channels: S 595389, P 162778
Pick format:
   id  phase_index                        timestamp   prob type
0   0         1460 2023-08-23 19:19:43.439000+00:00  0.475    S
1  10         1460 2023-08-23 19:19:43.439000+00:00  0.605    S
2  20         1467 2023-08-23 19:19:43.509000+00:00  0.586    S
3  30         1469 2023-08-23 19:19:43.529000+00:00  0.602    S
4  40         1469 2023-08-23 19:19:43.529000+00:00  0.598    S

estimate_eps would give 0.032 s; using 10 s
center: (-121.79093499999999, 46.76968)
xlim_degree: (-123, -

## 2. Association with GaMMA

In [4]:
import time
t0 = time.time()
event_idx0 = 0  ## current earthquake index
assignments = []
events, assignments = association(picks, stations, config, event_idx0, config["method"])
event_idx0 += len(events)
print(f"{len(events)} events from {len(assignments)} associated picks in {time.time() - t0:.0f} s")

## create catalog
events = pd.DataFrame(events)
if len(events):
    events[["longitude", "latitude"]] = events.apply(lambda x: pd.Series(proj(longitude=x["x(km)"], latitude=x["y(km)"], inverse=True)), axis=1)
    events["depth_km"] = events["z(km)"]
    events.to_csv(result_path("gamma_events.csv"), index=False,
                  float_format="%.3f",
                  date_format="%Y-%m-%dT%H:%M:%S.%f")

## add assignment to picks
assignments = pd.DataFrame(assignments, columns=["pick_index", "event_index", "gamma_score"])
picks = picks.join(assignments.set_index("pick_index")).fillna(-1).astype({"event_index": int})
picks.rename(columns={"id": "station_id", "timestamp": "phase_time", "type": "phase_type", "prob": "phase_score"}, inplace=True)
picks.to_csv(result_path("gamma_picks.csv"), index=False,
             date_format="%Y-%m-%dT%H:%M:%S.%f")

if len(events):
    n_by_phase = picks[picks["event_index"] >= 0].groupby(["event_index", "phase_type"]).size().unstack(fill_value=0)
    events = events.merge(n_by_phase.rename(columns={"P": "n_p", "S": "n_s"}), left_on="event_index", right_index=True, how="left")
    cols = ["event_index", "time", "longitude", "latitude", "depth_km", "num_picks", "n_p", "n_s", "sigma_time", "gamma_score"]
    events["time"] = pd.to_datetime(events["time"], utc=True)
    per_day = events.set_index("time").resample("1D").size()
    print(f"events per day: median {per_day.median():.0f}, max {per_day.max()} on {per_day.idxmax().date()}; "
          f"{(events['n_p'] == 0).sum()} S-only events, {(events['n_s'] == 0).sum()} P-only events")
    with pd.option_context("display.width", 200, "display.max_columns", 20):
        print(events[[c for c in cols if c in events.columns]].head(30).to_string(index=False))

Eikonal Solver: 
Iter 0, error = 999.828
Iter 1, error = 0.000
Time: 4.923
Eikonal Solver: 
Iter 0, error = 999.702
Iter 1, error = 0.000
Time: 0.024


Clustering (eps=8.33): 100%|██████████| 18985/18985 [00:41<00:00, 459.62it/s]


Associating 18984 clusters with 16 CPUs
................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

In [5]:
import pandas as pd

In [6]:
gamma_events = pd.read_csv('/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/gamma_association_11022026/rainier_mora_das/gamma_events.csv')
gamma_events

,time,magnitude,sigma_time,sigma_amp,cov_time_amp,gamma_score,num_picks,num_p_picks,num_s_picks,event_index,x(km),y(km),z(km),longitude,latitude,depth_km
0,2023-10-26T07:00:02.308,999,0.080,0,0,640.000,640,320,320,0,4.333,8.912,3.846,-121.734,46.850,3.846
1,2023-11-10T17:17:57.005,999,0.120,0,0,120.000,101,34,67,1,1.671,1.450,0.000,-121.769,46.783,0.000
2,2023-09-15T22:01:24.239,999,0.226,0,0,170.000,129,0,129,2,44.607,-5.791,19.724,-121.208,46.716,19.724
3,2023-11-10T16:08:22.280,999,0.082,0,0,351.273,414,171,243,13,6.611,-17.572,15.796,-121.705,46.612,15.796
4,2023-11-10T16:08:24.611,999,0.176,0,0,143.174,136,63,73,14,-3.803,-5.300,15.092,-121.841,46.722,15.092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
937,2023-09-10T05:59:06.014,999,0.058,0,0,133.915,134,0,134,928,1.235,4.019,5.072,-121.775,46.806,5.072
938,2023-08-27T11:46:16.532,999,0.377,0,0,466.000,390,193,197,930,-6.290,8.729,1.727,-121.873,46.848,1.727
939,2023-09-13T12:06:58.559,999,0.071,0,0,564.487,586,288,298,939,-10.723,1.292,9.112,-121.931,46.781,9.112
940,2023-08-25T14:29:31.041,999,0.239,0,0,216.852,196,98,98,937,-6.321,-2.942,4.946,-121.874,46.743,4.946
